In [1]:
import mlflow

mlflow.set_experiment("Credit Card Fraud Detection")

print("MLflow experiment configured.")

MLflow experiment configured.


In [2]:
mlflow.set_experiment("Credit Card Fraud Detection")

print("Experiment ready.")

Experiment ready.


In [3]:
with mlflow.start_run():
    print("MLflow run started.")

MLflow run started.


In [4]:
with mlflow.start_run():
    mlflow.log_params({
        "model":"XGBoost",
        "n_estimators":200,
        "max_depth":6,
        "learning_rate":0.1,
        "subsample":0.8,
        "colsample_bytree":0.8,
        "scale_pos_weight":599.4761904761905,
    })

    print("Paramaters logged")

Paramaters logged


In [5]:
with mlflow.start_run():
    mlflow.log_metrics({
        "precision":0.9048,
        "recall":0.8000,
        "f1_score":0.8492,
        "roc_auc":0.9761,
        "pr_auc":0.8248,
    })

    print("Metrics logged. ")

Metrics logged. 


In [6]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)



In [7]:
X_train = pd.read_csv("creditCardFraud/data/processed/X_train_scaled.csv")
y_train = pd.read_csv("creditCardFraud/data/processed/y_train.csv").squeeze("columns")

X_test = pd.read_csv("creditCardFraud/data/processed/X_test_scaled.csv")
y_test = pd.read_csv("creditCardFraud/data/processed/y_test.csv").squeeze("columns")

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(226980, 30) (226980,)
(56746, 30) (56746,)


In [8]:
scale_pos_weight = (y_train == 0).sum()/(y_train == 1).sum()

model = XGBClassifier(
    n_estimators = 200,
    max_depth = 6,
    learning_rate = 0.1,
    subsample = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = scale_pos_weight,
    objective = "binary:logistic",
    eval_metric = "logloss",
    random_state = 42,
    n_jobs = -1,
)

print(f"scale_pos_weight: {scale_pos_weight}")

scale_pos_weight: 599.4761904761905


In [15]:
from sklearn.metrics import average_precision_score, roc_auc_score

current_proba = model.predict_proba(X_test)[:, 1]

print("current PR-AUC:", average_precision_score(y_test, current_proba))
print("Current ROC-AUC:", roc_auc_score(y_test, current_proba))
print("Current probability range:", current_proba.min(), current_proba.max())

current PR-AUC: 0.8248384759168829
Current ROC-AUC: 0.9761103301934559
Current probability range: 4.4280807e-08 0.9999987


In [18]:
with mlflow.start_run(run_name = "Final XGBoost - verified") as run:

    y_proba = model.predict_proba(X_test)[:, 1]

    threshold = 0.2325
    y_pred = (y_proba >= threshold).astype(int)

    metrics = {
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    mlflow.log_params({
        "model": "XGBoost",
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight":scale_pos_weight,
        "threshold":threshold,
    })

    mlflow.log_metrics(metrics)

    mlflow.xgboost.log_model(
        model,
        name = "xgboost_model"
    )


    print("Run ID:", run.info.run_id)
    print(metrics)


Run ID: c04022290d2e49899579f565a9d4e667
{'precision': 0.9047619047619048, 'recall': 0.8, 'f1_score': 0.8491620111731844, 'roc_auc': 0.9761103301934559, 'pr_auc': 0.8248384759168829}


In [19]:
runs = mlflow.search_runs()

runs[[
"run_id",
"metrics.pr_auc",
"metrics.recall",
"metrics.precision",
"metrics.f1_score",
"params.model",
"params.threshold",
]]


,run_id,metrics.pr_auc,metrics.recall,metrics.precision,metrics.f1_score,params.model,params.threshold
0,c04022290d2e49899579f565a9d4e667,0.824838,0.8,0.904762,0.849162,XGBoost,0.2325
1,df248e97197e41fd99fea6317954275b,0.824838,0.8,0.904762,0.849162,XGBoost,0.2325
2,e848a18ff20546cba96424de57b7988f,0.724144,0.8,0.904762,0.849162,XGBoost,0.2325
3,60917b956c3f4c5fa6ca09406cdd4b4a,0.724144,0.8,0.904762,0.849162,XGBoost,0.2325
4,bf239d6bd3f0403d883d5471fa9c56b7,0.824800,0.8,0.904800,0.849200,None,None
5,fd847407c60541159c763c9b3025a440,NaN,NaN,NaN,NaN,XGBoost,None
6,31ed497a490346b9a5c2bd5c9a8eb9cf,NaN,NaN,NaN,NaN,None,None
7,7f727f7a649a458bb3ef65b5f977c033,0.824800,0.8,0.904800,0.849200,None,None
8,0f742daa97eb4b0baf25853b74292b7b,NaN,NaN,NaN,NaN,XGBoost,None
9,67d2b2e04b5641c3828be762cf65041b,NaN,NaN,NaN,NaN,None,None


In [20]:
import os
print("Notebook directory:", os.getcwd())
print("MLflow tracking URI:", mlflow.get_tracking_uri())

Notebook directory: C:\Users\hnkru
MLflow tracking URI: sqlite:///C:/Users/hnkru/mlflow.db
